- **Name:** 13.1_dataframe_aggregation
- **Author:** Shamas Imran
- **Desciption:** Aggregation operations on DataFrames
- **Date:** 19-Aug-2025
<!--
REVISION HISTORY
Version          Date        Author           Desciption
01           19-Aug-2025   Shamas Imran       Applied groupBy and agg functions  
                                              Demonstrated count, avg, max, min  
                                              Compared multiple aggregation styles  
-->

In [1]:
from pyspark.sql.types import *
from pyspark.sql import Row
from datetime import date
import random

# -------------------------------
# 1. Student DataFrame
# -------------------------------
student_schema = StructType([
    StructField('StudentID', IntegerType(), False),
    StructField('StudentName', StringType(), True),
    StructField('StudentAge', IntegerType(), True)
])

student_data = [
    (1, "Alice", 34), 
    (2, "Bob", 45), 
    (3, "Charlie", 29),
    (4, "Shamas", 40)
]

df_student = spark.createDataFrame(student_data, student_schema)

# -------------------------------
# 2. Course DataFrame
# -------------------------------
course_schema = StructType([
    StructField('CourseID', IntegerType(), False),
    StructField('CourseName', StringType(), True),
    StructField('CourseTitle', StringType(), True),
])

course_data = [
    (1, "Physics", "1111"), 
    (2, "Chemistry", "2222"), 
    (3, "English", "3333"),
    (4, "Computer Science", "4444")
]

df_course = spark.createDataFrame(course_data, course_schema)

# -------------------------------
# 3. Enrollment DataFrame
# -------------------------------
enrollment_schema = StructType([
    StructField("EnrollmentID", IntegerType(), False),
    StructField("StudentID_FK", IntegerType(), False),
    StructField("CourseID_FK", IntegerType(), False),
    StructField("EnrollmentDate", DateType(), True)
])

enrollment_data = [
    (1, 1, 1, date(2023, 9, 1)),   # Alice -> Physics
    (2, 2, 2, date(2023, 9, 2)),   # Bob -> Chemistry
    (3, 4, 4, date(2023, 9, 4)),   # Shamas -> Computer Science
    (4, 1, 2, date(2023, 9, 5)),   # Alice -> Chemistry
]

df_enrollment = spark.createDataFrame(enrollment_data, enrollment_schema)

# -------------------------------
# 4. Score DataFrame
# -------------------------------
semesters = ["2023-Spring", "2023-Fall", "2024-Spring", "2024-Fall", "2025-Spring"]
score_data = []
score_id = 1
enrollment_ids = [row.EnrollmentID for row in df_enrollment.collect()]

for enrollment_id in enrollment_ids:
    selected_semesters = random.sample(semesters, k=random.randint(2, 4))
    for sem in selected_semesters:
        score_data.append(Row(
            ScoreID=score_id,
            EnrollmentID_FK=enrollment_id,
            Semester=sem,
            Score=random.randint(60, 100)
        ))
        score_id += 1

score_schema = StructType([
    StructField("ScoreID", IntegerType(), False),
    StructField("EnrollmentID_FK", IntegerType(), False),
    StructField("Semester", StringType(), True),
    StructField("Score", IntegerType(), True)
])

df_score = spark.createDataFrame(score_data, schema=score_schema)

StatementMeta(, 4202b6d1-e506-4b50-8f9c-01ec09482984, 3, Finished, Available, Finished)

In [2]:
from pyspark.sql import functions as F

# 1. Count of enrollments per student
df_enrollment.groupBy("StudentID_FK") \
    .agg(F.count("*").alias("TotalEnrollments")) \
    .show()

StatementMeta(, 4202b6d1-e506-4b50-8f9c-01ec09482984, 4, Finished, Available, Finished)

+------------+----------------+
|StudentID_FK|TotalEnrollments|
+------------+----------------+
|           1|               2|
|           2|               1|
|           4|               1|
+------------+----------------+



In [3]:
# 2. Average score per student
df_score.alias("s") \
    .join(df_enrollment.alias("e"), F.col("s.EnrollmentID_FK") == F.col("e.EnrollmentID")) \
    .groupBy("e.StudentID_FK") \
    .agg(F.avg("s.Score").alias("AvgScore")) \
    .show()

StatementMeta(, 4202b6d1-e506-4b50-8f9c-01ec09482984, 5, Finished, Available, Finished)

+------------+--------+
|StudentID_FK|AvgScore|
+------------+--------+
|           1|    87.0|
|           4|    75.0|
|           2|    73.0|
+------------+--------+



In [0]:
# 3. Best score per semester
df_score.groupBy("Semester") \
    .agg(F.max("Score").alias("BestScore")) \
    .show()

In [0]:
# 4. Number of unique students per course
df_enrollment.alias("e") \
    .join(df_student.alias("st"), F.col("e.StudentID_FK") == F.col("st.StudentID")) \
    .join(df_course.alias("c"), F.col("e.CourseID_FK") == F.col("c.CourseID")) \
    .groupBy("c.CourseName") \
    .agg(F.countDistinct("st.StudentID").alias("UniqueStudents")) \
    .show()